# Experiment v4 — Isolation Phase (Jupyter server)

Single-turn isolation experiment for the EMCTS family on a 100-scenario portfolio. Designed for the
120-core / 1 TB server, but RAM-light: `-Xmx32g` is plenty.

**Prerequisites**
1. Repo cloned to the server (`git clone` of Bachelor_Thesis, branch `experiment-version-4`).
2. JDK 8+ and Python 3 with pip available (`!java -version`, `!python --version`).
3. Run from the `Tribes-master/` directory (set the path below).

**Notes**
- Linux classpath separator is `:` (Windows uses `;`).
- The full run is roughly 1.5–2 h with overhead (~50 min idealised). Run it via `nohup` so it survives
  a kernel/browser disconnect; it is also fully resumable per (agent, ratio, ms).
- The 12-config × 5000-turn 5000 ms pass is the long pole.

In [ ]:
import os
# EDIT this to the absolute path of the Tribes-master directory on the server:
REPO = os.path.expanduser('~/Bachelor_Thesis/Tribes-master')
os.chdir(REPO)
print('cwd =', os.getcwd())
print(os.listdir('.'))   # expect src/, lib/, analysis/ ...

In [ ]:
# 0. Environment check
!java -version
!python --version
!nproc                 # number of cores
!free -g | head -2     # RAM in GB

## 1. Compile

In [ ]:
!rm -rf out && mkdir -p out
!find src -name '*.java' > _srcs.txt
!javac -d out -cp "lib/*" @_srcs.txt && echo 'BUILD OK' || echo 'BUILD FAILED'

## 2. Smoke test (~2 s) — pipeline correctness
Verifies FastGameStateIO round-trip, the shared random initial sequence across agents (the heart of
v4's paired methodology), folder layout, and CSV schema. If this fails, do NOT launch the full run.

In [ ]:
!java -Xmx4g -cp "out:lib/*" core.data.exp.NewExperimentSmokeTestV4

## 3. Run v4 (background + log)
Starts the full run detached so it survives a disconnect. `tail` the log to monitor; the run is
resumable per (agent, ratio, ms) if interrupted.

Args: `<threads> [<msCsv>]`. Default ms levels are `50,500,5000`.

32 GB heap is comfortable for 120 in-flight tasks (each task holds one GameState deep-copy).

In [ ]:
!nohup java -Xmx32g -cp "out:lib/*" core.data.exp.RunExperimentV4 120 > v4_run.log 2>&1 &
print('v4 started in background -> v4_run.log')

In [ ]:
# Monitor progress (re-run to refresh)
!tail -n 50 v4_run.log
!echo '--- java processes ---'; pgrep -af RunExperimentV4 || echo '(none - finished or not started)'

## 4. Analyse (after the run completes)
Output goes to `analysis/figures_v4/` and `analysis/tables_v4/` (kept separate from v1–v3 analyses).

Headline figure: `fig1_paired_delta_vs_emcts.png` — paired (per scenario × visit) Δ FinalSequenceValue
for every config vs EMCTS, with 95 % CIs. This is the comparison v4 exists to enable.

In [ ]:
!python -m pip install -q pandas matplotlib numpy
!python analysis/analyze_v4.py test_results/v4

In [ ]:
import glob
from IPython.display import Image, display
for f in sorted(glob.glob('analysis/figures_v4/*.png')):
    print(f); display(Image(filename=f))